In [ ]:
# Keep Kaggle output readable: constant genes are ranked last by the F-test implementation.
import warnings
warnings.filterwarnings('ignore', message=r'Features .* are constant\.', category=UserWarning)
warnings.filterwarnings('ignore', message=r'invalid value encountered in divide', category=RuntimeWarning)


# E2 + E3 v3 — benchmark chi phí chọn gene

E2 và E3 chạy chung vì cùng đo feature selection trên một cohort, một split và một máy. E2 đo điểm chuẩn cố định; E3 thay đổi n/p hoặc budget wrapper. Không chạy classifier trong notebook này.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

# Set before importing NumPy/scikit-learn. Run this notebook alone in its Kaggle session.
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'
INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')

COHORT = 'tcga_luad_lusc_primary_tumor'
E2_K_GRID = [10, 50]
E2_REPEATS = 3
E3_K = 50
E3_N_GRID = [100, 200, 300, None]
E3_P_GRID = [1000, 2000, 5000, 10000, None]
NON_WRAPPER_METHODS = ['f_test', 'mutual_info', 'mrmr', 'relieff', 'l1_lr', 'elastic_net', 'rf_importance', 'svm_rfe']
WRAPPER_METHODS = ['ga', 'bpso']
E3_WRAPPER_BUDGETS = [(10, 20), (30, 50)]


In [ ]:
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
RUNTIME_IMPORTS = ('numpy', 'pandas', 'scipy', 'sklearn', 'tables', 'joblib', 'threadpoolctl', 'psutil')
try:
    for package in RUNTIME_IMPORTS:
        print(package, __import__(package).__version__)
except Exception as exc:
    raise RuntimeError('Restart the Kaggle session and rerun from Cell 1; do not install requirements.txt.') from exc
# ReliefF is the only missing package on Kaggle; --no-deps preserves its compatible numeric stack.
try:
    import skrebate
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'skrebate==0.8.4'], check=True)
    import skrebate
print('skrebate', skrebate.__version__)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    source, target = INPUT_V3_DATASET / folder, WORK_ROOT / folder
    assert source.is_dir(), f'Missing Kaggle input: {source}'
    if not target.exists():
        target.symlink_to(source, target_is_directory=True)
os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('commit:', COMMIT)


In [ ]:
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from survey_v2 import config as C
from survey_v2.data import load_dataset, sha256, variance_prefilter
from survey_v2.e1_splits import validate_repeated_split_manifest
from survey_v2.methods import select
from survey_v2.timing import empirical_exponent, machine_info, measure, subsample_rows, top_variance_columns
from scripts.run_e1_feature_selection import split_indices

def atomic_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def timed_selection(method, X, y, k, seed, wrapper_kwargs=None):
    def run(X_arg, y_arg, k_arg):
        with threadpool_limits(1):
            return select(method, X_arg, y_arg, [k_arg], seed=seed, n_jobs=1, wrapper_kwargs=wrapper_kwargs)
    return measure(run, X, y, k, repeats=E2_REPEATS)

ds = load_dataset(WORK_ROOT, COHORT)
manifest_path = ds.data_dir / 'repeated_splits.csv'
manifest = pd.read_csv(manifest_path)
errors = validate_repeated_split_manifest(manifest, n_folds=C.N_FOLDS)
assert not errors, errors
train, _ = split_indices(ds, manifest, repeat=0, fold=0)
seed = int(manifest[(manifest.repeat == 0) & (manifest.fold == 0)].seed.iloc[0])
X_train, y_train = ds.X[train], ds.y[train]
output_root = WORK_ROOT / 'results' / 'survey_v3' / 'E2_E3'
(output_root / 'machine_config.json').parent.mkdir(parents=True, exist_ok=True)
(output_root / 'machine_config.json').write_text(json.dumps(machine_info(), indent=2), encoding='utf-8')
print({'cohort': COHORT, 'n_train': len(train), 'p_raw': X_train.shape[1], 'machine': machine_info()})


In [ ]:
# E2: fixed point, p=5,000 after train-only variance prefilter, k={10,50}.
rows = []
pre = variance_prefilter(X_train, 5000)
for method_id, method in enumerate(NON_WRAPPER_METHODS):
    for k in E2_K_GRID:
        result = timed_selection(method, X_train[:, pre], y_train, k, C.seed_for(C.dataset_seed_id(COHORT), seed, method_id, k))
        rows.append({'experiment': 'E2', 'cohort': COHORT, 'method': method, 'k': k, 'n': len(train), 'p': len(pre), 'wrapper_population': None, 'wrapper_generations': None, **result})
        print(f"E2 {method} k={k}: {result['wall_s']:.2f}s", flush=True)
for method_id, method in enumerate(WRAPPER_METHODS, start=len(NON_WRAPPER_METHODS)):
    for k in E2_K_GRID:
        result = timed_selection(method, X_train[:, pre], y_train, k, C.seed_for(C.dataset_seed_id(COHORT), seed, method_id, k), {'pop': 30, 'generations': 50})
        rows.append({'experiment': 'E2', 'cohort': COHORT, 'method': method, 'k': k, 'n': len(train), 'p': len(pre), 'wrapper_population': 30, 'wrapper_generations': 50, **result})
        print(f"E2 {method} k={k}: {result['wall_s']:.2f}s", flush=True)
atomic_csv(pd.DataFrame(rows), output_root / 'e2_cost_point.csv')


In [ ]:
# E3: empirical scaling. Non-wrappers vary n and p; wrappers vary evaluation budget.
n_rows, p_rows, wrapper_rows = [], [], []
for n in E3_N_GRID:
    idx = subsample_rows(y_train, n, np.random.default_rng([C.dataset_seed_id(COHORT), seed, 3, 0 if n is None else n]))
    cols = variance_prefilter(X_train[idx], 5000)
    for method_id, method in enumerate(NON_WRAPPER_METHODS):
        result = timed_selection(method, X_train[idx][:, cols], y_train[idx], E3_K, C.seed_for(C.dataset_seed_id(COHORT), seed, 3, method_id, len(idx)))
        n_rows.append({'cohort': COHORT, 'method': method, 'k': E3_K, 'n': len(idx), 'p': len(cols), **result})
for p in E3_P_GRID:
    cols = top_variance_columns(X_train, p)
    for method_id, method in enumerate(NON_WRAPPER_METHODS):
        result = timed_selection(method, X_train[:, cols], y_train, E3_K, C.seed_for(C.dataset_seed_id(COHORT), seed, 4, method_id, len(cols)))
        p_rows.append({'cohort': COHORT, 'method': method, 'k': E3_K, 'n': len(train), 'p': len(cols), **result})
for method_id, method in enumerate(WRAPPER_METHODS):
    for pop, generations in E3_WRAPPER_BUDGETS:
        result = timed_selection(method, X_train[:, pre], y_train, E3_K, C.seed_for(C.dataset_seed_id(COHORT), seed, 5, method_id, pop, generations), {'pop': pop, 'generations': generations})
        wrapper_rows.append({'cohort': COHORT, 'method': method, 'k': E3_K, 'n': len(train), 'p': len(pre), 'population': pop, 'generations': generations, 'expected_evaluations': pop * generations, **result})
n_df, p_df = pd.DataFrame(n_rows), pd.DataFrame(p_rows)
atomic_csv(n_df, output_root / 'e3_scaling_n.csv')
atomic_csv(p_df, output_root / 'e3_scaling_p.csv')
atomic_csv(pd.DataFrame(wrapper_rows), output_root / 'e3_wrapper_evals.csv')
exponents = []
for method in NON_WRAPPER_METHODS:
    exponents.append({'cohort': COHORT, 'method': method, 'n_exponent': empirical_exponent(n_df[n_df.method == method], 'n'), 'p_exponent': empirical_exponent(p_df[p_df.method == method], 'p')})
atomic_csv(pd.DataFrame(exponents), output_root / 'e3_exponents.csv')


In [ ]:
config = {'experiment': 'E2_E3_v3', 'cohort': COHORT, 'outer_train_split': {'repeat': 0, 'fold': 0, 'split_sha256': sha256(manifest_path)}, 'e2': {'k_grid': E2_K_GRID, 'p': 5000, 'repeats': E2_REPEATS, 'wrapper_budget': [30, 50]}, 'e3': {'k': E3_K, 'n_grid': E3_N_GRID, 'p_grid': E3_P_GRID, 'wrapper_budgets': E3_WRAPPER_BUDGETS}, 'blas_threads': 1, 'commit_hash': COMMIT, 'traceable': True}
(output_root / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
archive = Path('/kaggle/working') / 'e2_e3_tcga_luad_lusc.zip'
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK_ROOT, base_dir='results/survey_v3/E2_E3')
print('DONE:', archive)
